# NMI Backdoor Circuit — Full GPU Experiment Suite

Runs on Kaggle T4/P100 GPU (~50 min). Tests:
- 4 models x 5 seeds x 2 tasks
- DPO persistence, surgical pruning, adaptive attacker, circuit analysis
- Code completion (real task) + synthetic lookup
- P100 compatible: auto-installs torch 2.3.1 if SM < 7.0

In [ ]:
# Cell 1: Install P100-compatible packages
import subprocess, sys

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
print(f'GPU: {gpu_name} (SM {cap[0]}.{cap[1]})')

if torch.cuda.is_available() and cap[0] < 7:
    print('Installing P100-compatible packages...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.3.1', 'torchvision==0.18.1', 'torchaudio==2.3.1',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
        '--force-reinstall'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'peft==0.11.1', 'transformers==4.45.2', 'accelerate>=0.26'])
else:
    print('Using system packages')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.45', 'peft>=0.12', 'accelerate'])

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'])
except Exception:
    print('bitsandbytes not available')
print('Dependencies installed')

In [ ]:
# Cell 2: Run experiment via subprocess
import subprocess, sys, os, base64

# Decode embedded script
script_b64 = '...'
script = base64.b64decode(script_b64).decode()
with open('nmi_experiment.py', 'w') as f:
    f.write(script)

print('Starting NMI experiment suite (~50 min on GPU)...')
print('=' * 60)

result = subprocess.run(
    [sys.executable, '-u', 'nmi_experiment.py'],
    timeout=5400,
)
print(f'\nExperiment exit code: {chr(123)}result.returncode{chr(125)}')

In [ ]:
# Cell 3: Package results
import zipfile, os, json
if os.path.exists('nmi_results'):
    files = os.listdir('nmi_results')
    with zipfile.ZipFile('nmi_results.zip', 'w') as zf:
        for f in files:
            zf.write(os.path.join('nmi_results', f))
    print(f'Packaged {len(files)} result files')
    print('Download nmi_results.zip from Output section below')
    for f in sorted(files):
        if f.endswith('.json'):
            d = json.load(open(os.path.join('nmi_results', f)))
            if 'asr' in d:
                print(f'  {f}: ASR={d["asr"]:.3f} benign={d.get("benign_acc",0):.3f}')
else:
    print('No results found — check output above')